<a href="https://colab.research.google.com/github/manika-17/dsa405-project/blob/main/DSA405_P2_AuditCleaningLog_FA26_mjain23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Data Audit
The goal of this audit is to systematically inspect the raw Best Picture data before making cleaning decisions. I will examine the columns used in the analysis, their data types, missingness, distinct values, ranges, and categorical values. I will also explicitly check for common defects such as duplicate keys, sentinel values, total/subtotal rows, encoding problems, inconsistent year formats, and numbers stored as text.

In [15]:
import pandas as pd
import numpy as np
import re
import requests
from io import StringIO

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

URL = "https://en.wikipedia.org/wiki/Academy_Award_for_Best_Picture"

headers = {
    "User-Agent": "NCSU DSA405 student course project; educational/non-commercial use"
}

response = requests.get(URL, headers=headers, timeout=30)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))

print("HTTP status:", response.status_code)
print("Number of HTML tables found:", len(tables))

HTTP status: 200
Number of HTML tables found: 23


Locate the award tables
Wikipedia's article contains multiple HTML tables. The following code identifies tables containing a Film column and inspects their structure before any records are changed.

In [16]:
candidate_tables = []

for i, table in enumerate(tables):
    flat_cols = [
        " ".join(map(str, col)).strip() if isinstance(col, tuple) else str(col).strip()
        for col in table.columns
    ]
    if any("Film" in col for col in flat_cols):
        candidate_tables.append((i, table.copy(), flat_cols))

print("Candidate tables containing a Film column:", len(candidate_tables))

for i, table, flat_cols in candidate_tables:
    print("\nTABLE", i, "| rows:", len(table), "| columns:", len(table.columns))
    print(flat_cols)
    display(table.head(3))

Candidate tables containing a Film column: 13

TABLE 1 | rows: 28 | columns: 5
['Year', 'Best Picture Winner', 'Best Picture Director', 'Best Director Film', 'Best Director Winner']


,Year,Best Picture Winner,Best Picture Director,Best Director Film,Best Director Winner
0,1927-28,Wings,William A. Wellman,7th Heaven (Dramatic Picture),Frank Borzage
1,1927-28,Wings,William A. Wellman,Two Arabian Knights (Comedy Picture),Lewis Milestone
2,1928-29,The Broadway Melody,Harry Beaumont,The Divine Lady,Frank Lloyd



TABLE 3 | rows: 8 | columns: 3
['Year of film release', 'Film', 'Film studio']


,Year of film release,Film,Film studio
0,1927/28 (1st),Wings,"Paramount (Lucien Hubbard, Jesse L. Lasky, B. ..."
1,1927/28 (1st),7th Heaven,"Fox (William Fox, producer)"
2,1927/28 (1st),The Racket,"The Caddo Company (Howard Hughes, producer)"



TABLE 4 | rows: 92 | columns: 3
['Year of film release', 'Film', 'Film studio/Producer(s)']


...

,Year of film release,Film,Producer(s)
0,2010 (83rd),The King's Speech,"Iain Canning, Emile Sherman, and Gareth Unwin"
1,2010 (83rd),Black Swan,"Scott Franklin, Mike Medavoy, and Brian Oliver"
2,2010 (83rd),The Fighter,"David Hoberman, Todd Lieberman, and Mark Wahlberg"



TABLE 13 | rows: 58 | columns: 3
['Year of film release', 'Film', 'Producer(s)']


,Year of film release,Film,Producer(s)
0,2020/21 [f] (93rd),Nomadland,"Frances McDormand, Peter Spears, Mollye Asher,..."
1,2020/21 [f] (93rd),The Father,"David Parfitt, Jean-Louis Livi, and Philippe C..."
2,2020/21 [f] (93rd),Judas and the Black Messiah,"Shaka King, Charles D. King, and Ryan Coogler"



TABLE 14 | rows: 4 | columns: 4
['Record', 'Producer', 'Film', 'Age']


,Record,Producer,Film,Age
0,Oldest winner,Saul Zaentz,The English Patient,"76 years, 24 days"
1,Oldest nominee,Clint Eastwood,American Sniper,"84 years, 229 days"
2,Youngest winner,Carl Laemmle Jr.,All Quiet on the Western Front,"22 years, 191 days"


### Build the raw project extract
The exact Wikipedia table structure can change over time. The code below standardizes column labels, keeps tables that contain the Best Picture film records, and combines them. It does not overwrite the downloaded HTML or modify the source.

In [17]:
def flatten_columns(df):
    out = df.copy()
    out.columns = [
        " ".join([str(x) for x in col if str(x) != "nan"]).strip()
        if isinstance(col, tuple)
        else str(col).strip()
        for col in out.columns
    ]
    return out

award_tables = []

for i, table, _ in candidate_tables:
    t = flatten_columns(table)

    film_cols = [c for c in t.columns if re.search(r"\bFilm\b", c, flags=re.I)]
    year_cols = [c for c in t.columns if re.search(r"Year", c, flags=re.I)]

    if film_cols and year_cols:
        award_tables.append((i, t))

print("Award tables retained:", [i for i, _ in award_tables])

if not award_tables:
    raise ValueError("No award tables with both Film and Year columns were detected.")

raw_parts = []

for table_number, t in award_tables:
    film_col = next(c for c in t.columns if re.search(r"\bFilm\b", c, flags=re.I))
    year_col = next(c for c in t.columns if re.search(r"Year", c, flags=re.I))

    part = t.copy()
    part["_source_table"] = table_number
    part["_film_source_col"] = film_col
    part["_year_source_col"] = year_col
    raw_parts.append(part)

raw_combined = pd.concat(raw_parts, ignore_index=True, sort=False)

print("Rows in combined raw award tables:", len(raw_combined))
print("Columns in combined raw award tables:", len(raw_combined.columns))
display(raw_combined.head())

Award tables retained: [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Rows in combined raw award tables: 649
Columns in combined raw award tables: 13


,Year,Best Picture Winner,Best Picture Director,Best Director Film,Best Director Winner,_source_table,_film_source_col,_year_source_col,Year of film release,Film,Film studio,Film studio/Producer(s),Producer(s)
0,1927-28,Wings,William A. Wellman,7th Heaven (Dramatic Picture),Frank Borzage,1,Best Director Film,Year,NaN,NaN,NaN,NaN,NaN
1,1927-28,Wings,William A. Wellman,Two Arabian Knights (Comedy Picture),Lewis Milestone,1,Best Director Film,Year,NaN,NaN,NaN,NaN,NaN
2,1928-29,The Broadway Melody,Harry Beaumont,The Divine Lady,Frank Lloyd,1,Best Director Film,Year,NaN,NaN,NaN,NaN,NaN
3,1930-31,Cimarron,Wesley Ruggles,Skippy,Norman Taurog,1,Best Director Film,Year,NaN,NaN,NaN,NaN,NaN
4,1931-32,Grand Hotel,Edmund Goulding,Bad Girl,Frank Borzage,1,Best Director Film,Year,NaN,NaN,NaN,NaN,NaN


### Create a minimally structured audit table
This step creates consistent film_year and title fields so that the relevant 2015–2024 records can be audited. The original combined table remains available as raw_combined.

In [18]:
audit_parts = []

for table_number, t in award_tables:
    film_col = next(c for c in t.columns if re.search(r"\bFilm\b", c, flags=re.I))
    year_col = next(c for c in t.columns if re.search(r"Year", c, flags=re.I))

    x = pd.DataFrame({
        "film_year_raw": t[year_col],
        "title_raw": t[film_col],
        "source_table": table_number
    })

    # Wikipedia often displays a year once and leaves subsequent nominee rows blank.
    x["film_year_raw"] = x["film_year_raw"].ffill()
    audit_parts.append(x)

audit_raw = pd.concat(audit_parts, ignore_index=True)

# Extract the first four-digit year from the displayed year field.
audit_raw["film_year"] = pd.to_numeric(
    audit_raw["film_year_raw"].astype(str).str.extract(r"(\d{4})")[0],
    errors="coerce"
)

project_raw = audit_raw.loc[
    audit_raw["film_year"].between(2015, 2024, inclusive="both")
].copy()

print("Rows in project period before cleaning:", len(project_raw))
print("Columns in project audit extract:", len(project_raw.columns))
display(project_raw.head(15))

Rows in project period before cleaning: 95
Columns in project audit extract: 4


,film_year_raw,title_raw,source_table,film_year
24,2015,The Revenant,1,2015
25,2016,La La Land,1,2016
26,2018,Roma,1,2018
27,2021,The Power of the Dog,1,2021
548,2015 (88th),2015 (88th),12,2015
549,2015 (88th),2015 (88th),12,2015
550,2015 (88th),2015 (88th),12,2015
551,2015 (88th),2015 (88th),12,2015
552,2015 (88th),2015 (88th),12,2015
553,2015 (88th),2015 (88th),12,2015


Systematic column audit

In [19]:
audit_summary = pd.DataFrame({
    "dtype_loaded": project_raw.dtypes.astype(str),
    "missing_count": project_raw.isna().sum(),
    "missing_rate": project_raw.isna().mean(),
    "distinct_values": project_raw.nunique(dropna=False)
})

audit_summary

,dtype_loaded,missing_count,missing_rate,distinct_values
film_year_raw,object,0,0.0,14
title_raw,object,0,0.0,14
source_table,int64,0,0.0,3
film_year,int64,0,0.0,10


In [20]:
print("film_year minimum:", project_raw["film_year"].min())
print("film_year maximum:", project_raw["film_year"].max())
print("film_year range:", project_raw["film_year"].max() - project_raw["film_year"].min())

for col in project_raw.columns:
    nunique = project_raw[col].nunique(dropna=False)
    if nunique <= 30:
        print(f"\nDistinct values for {col} ({nunique}):")
        print(project_raw[col].value_counts(dropna=False))

film_year minimum: 2015
film_year maximum: 2024
film_year range: 9

Distinct values for film_year_raw (14):
film_year_raw
2023 (96th)           10
2024 (97th)           10
2021 [g] (94th)       10
2022 (95th)           10
2016 (89th)            9
2019 (92nd)            9
2017 (90th)            9
2015 (88th)            8
2020/21 [f] (93rd)     8
2018 (91st)            8
2018                   1
2021                   1
2015                   1
2016                   1
Name: count, dtype: int64

Distinct values for title_raw (14):
title_raw
2023 (96th)             10
2024 (97th)             10
2021 [g] (94th)         10
2022 (95th)             10
2016 (89th)              9
2019 (92nd)              9
2017 (90th)              9
2015 (88th)              8
2020/21 [f] (93rd)       8
2018 (91st)              8
Roma                     1
The Power of the Dog     1
The Revenant             1
La La Land               1
Name: count, dtype: int64

Distinct values for source_table (3):
source_table

### Explicit defect checks
A clean result is still a finding. The following checks document both defects found and defect classes checked for but not found.

In [21]:
def text_contains_sentinel(series):
    sentinels = {"999", "-999", "n/a", "na", "null", "none", "missing", "unknown", ""}
    s = series.astype("string").str.strip().str.lower()
    return s.isin(sentinels)

checks = {}

checks["exact_duplicate_rows"] = int(project_raw.duplicated().sum())
checks["duplicate_title_year_keys_raw"] = int(
    project_raw.duplicated(subset=["film_year", "title_raw"]).sum()
)
checks["missing_titles"] = int(project_raw["title_raw"].isna().sum())
checks["missing_film_years"] = int(project_raw["film_year"].isna().sum())
checks["sentinel_like_titles"] = int(text_contains_sentinel(project_raw["title_raw"]).sum())

title_text = project_raw["title_raw"].astype("string")
checks["leading_or_trailing_title_whitespace"] = int(
    (title_text != title_text.str.strip()).fillna(False).sum()
)

checks["total_or_subtotal_title_rows"] = int(
    title_text.str.contains(r"^(total|subtotal)$", case=False, na=False).sum()
)

checks["non_2015_2024_years_after_filter"] = int(
    (~project_raw["film_year"].between(2015, 2024)).sum()
)

# Replacement-character check is a simple indicator of possible decoding/encoding damage.
checks["replacement_character_in_title"] = int(
    title_text.str.contains("�", regex=False, na=False).sum()
)

defect_checks = pd.Series(checks, name="count")
defect_checks

/tmp/ipykernel_552/529864746.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  title_text.str.contains(r"^(total|subtotal)$", case=False, na=False).sum()


,count
exact_duplicate_rows,81
duplicate_title_year_keys_raw,81
missing_titles,0
missing_film_years,0
sentinel_like_titles,0
leading_or_trailing_title_whitespace,0
total_or_subtotal_title_rows,0
non_2015_2024_years_after_filter,0
replacement_character_in_title,0


### Audit conclusion
The Wikipedia source is expected to be relatively well curated, but its presentation is designed for human readers rather than direct analysis. The main preparation needs are to restrict the data to film years 2015–2024, standardize useful column names, identify winner status consistently, retain only fields needed for the project, and create a standardized title field for later IMDb matching.
The code above also checks for missing values, exact duplicates, duplicate title-year keys, sentinel-like values, total/subtotal rows, leading or trailing title whitespace, unexpected years, and a common indicator of text-encoding damage. The displayed counts are the evidence for which defects were or were not found.

### 2. Cleaning Execution
The following transformations create the cleaned P2 dataset while preserving the raw source objects. Counts are captured as the transformations occur so that the cleaning log can report exact affected rows/cells.

In [22]:
clean = project_raw.copy()

log_rows = []

def add_log(column, change, affected, why, lost):
    log_rows.append({
        "#": len(log_rows) + 1,
        "Column": column,
        "Change made": change,
        "Rows/cells affected": int(affected),
        "Why": why,
        "What is lost": lost
    })

# 1. Restricting to 2015–2024 occurred when project_raw was created.
outside_period = int((~audit_raw["film_year"].between(2015, 2024, inclusive="both")).fillna(True).sum())
add_log(
    "film_year",
    "Filtered records to film years 2015–2024",
    outside_period,
    "The research question is explicitly limited to these ten film years.",
    "Award records outside the study period are excluded from the project extract."
)

# 2. Standardize title whitespace.
before_title = clean["title_raw"].astype("string")
after_title = before_title.str.strip().str.replace(r"\s+", " ", regex=True)
changed_ws = int(((before_title != after_title) & before_title.notna()).sum())
clean["title"] = after_title
add_log(
    "title",
    "Trimmed leading/trailing whitespace and collapsed repeated whitespace",
    changed_ws,
    "Consistent title text reduces avoidable mismatches during later IMDb matching.",
    "Only non-substantive whitespace formatting; the original title_raw field is retained during P2."
)

# 3. Convert film year to integer after filtering missing years.
missing_year_rows = int(clean["film_year"].isna().sum())
if missing_year_rows:
    clean = clean.loc[clean["film_year"].notna()].copy()
add_log(
    "film_year",
    "Removed records without a usable four-digit film year",
    missing_year_rows,
    "A film year is required to define the study period and later match titles.",
    "Any records lacking a usable film year would be excluded."
)

clean["film_year"] = clean["film_year"].astype(int)

# 4. Remove missing titles if present.
missing_title_rows = int(clean["title"].isna().sum() + clean["title"].eq("").sum())
if missing_title_rows:
    clean = clean.loc[clean["title"].notna() & clean["title"].ne("")].copy()
add_log(
    "title",
    "Removed records without a usable film title",
    missing_title_rows,
    "A title is required for the planned one-to-one match to IMDb.",
    "Records without a usable film identifier would be excluded."
)

# 5. Exact duplicate removal.
exact_dupes = int(clean.duplicated(subset=["film_year", "title"]).sum())
if exact_dupes:
    clean = clean.drop_duplicates(subset=["film_year", "title"], keep="first").copy()
add_log(
    "film_year, title",
    "Removed exact duplicate title-year records",
    exact_dupes,
    "Each nomination should appear once in the nominee-level analysis table.",
    "Repeated copies of the same title-year record only; one copy is retained."
)

# 6. Create award year.
clean["award_year"] = clean["film_year"] + 1
add_log(
    "award_year",
    "Created ceremony year as film_year + 1",
    len(clean),
    "The ceremony generally occurs in the calendar year following the film year and this field makes the distinction explicit.",
    "Nothing; this is a derived variable and film_year is retained."
)

# 7. Create title_clean for later matching.
clean["title_clean"] = (
    clean["title"]
    .str.normalize("NFKC")
    .str.lower()
    .str.replace(r"[’‘`]", "'", regex=True)
    .str.replace(r"[^a-z0-9]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
title_clean_changed = int((clean["title"].str.lower() != clean["title_clean"]).sum())
add_log(
    "title_clean",
    "Created lowercase alphanumeric standardized title for later matching",
    len(clean),
    "A separate standardized key will help resolve punctuation and formatting differences when joining to IMDb in P3.",
    "Nothing from the source because the original title field is retained."
)

# Winner detection:
# Wikipedia commonly marks winners using formatting/notes that may survive read_html differently.
# First inspect duplicate title-year patterns and source text; then use table ordering convention:
# in each year, the winning film is listed first in the Wikipedia award table.
clean = clean.sort_index()
clean["winner"] = ~clean["film_year"].duplicated(keep="first")
add_log(
    "winner",
    "Created Boolean winner indicator from the first listed film within each film year",
    len(clean),
    "Wikipedia's Best Picture tables list the winning film first for each award year; the project needs winner status for the comparison.",
    "Nothing; winner status is derived and source title/year fields are retained."
)

# 9. Drop source-only helper columns from final clean table.
final_cols = ["film_year", "award_year", "title", "title_clean", "winner"]
columns_before_final = clean.shape[1]
dropped_cols = [c for c in clean.columns if c not in final_cols]
clean_final = clean[final_cols].copy()

add_log(
    "columns",
    "Dropped source/helper fields from the final analytical table",
    len(dropped_cols),
    "The final P2 file keeps only variables needed for interpretation or the planned P3 match.",
    "Source-table and raw parsing fields are not carried into the final analytical table; the raw source remains preserved separately."
)

# 10. Reset index.
old_index = clean_final.index.copy()
clean_final = clean_final.reset_index(drop=True)
add_log(
    "index",
    "Reset pandas row index after filtering and cleaning",
    len(clean_final),
    "A sequential index makes the cleaned table easier to inspect and does not serve as an analytical identifier.",
    "Only the temporary pandas index; no source variable is removed."
)

print("Cleaned rows:", len(clean_final))
print("Cleaned columns:", clean_final.shape[1])
display(clean_final.head(15))

Cleaned rows: 14
Cleaned columns: 5


,film_year,award_year,title,title_clean,winner
0,2015,2016,The Revenant,the revenant,True
1,2016,2017,La La Land,la la land,True
2,2018,2019,Roma,roma,True
3,2021,2022,The Power of the Dog,the power of the dog,True
4,2015,2016,2015 (88th),2015 88th,False
5,2016,2017,2016 (89th),2016 89th,False
6,2017,2018,2017 (90th),2017 90th,True
7,2018,2019,2018 (91st),2018 91st,False
8,2019,2020,2019 (92nd),2019 92nd,True
9,2020,2021,2020/21 [f] (93rd),2020 21 f 93rd,True


### Winner-status validation
Because winner status is central to the research question, inspect the number of winners per film year. There should be exactly one winner in each of the ten selected years.

In [23]:
winner_validation = (
    clean_final.groupby("film_year")
    .agg(
        nominees=("title", "size"),
        winners=("winner", "sum")
    )
)

display(winner_validation)

assert winner_validation.index.min() == 2015
assert winner_validation.index.max() == 2024
assert len(winner_validation) == 10
assert (winner_validation["winners"] == 1).all(), "Winner detection did not produce exactly one winner per year."

print("Validation passed: exactly one winner is identified for every film year 2015–2024.")

,nominees,winners
film_year,,
2015,2,1
2016,2,1
2017,1,1
2018,2,1
2019,1,1
2020,1,1
2021,2,1
2022,1,1
2023,1,1


Validation passed: exactly one winner is identified for every film year 2015–2024.


### 3. Data Dictionary
The dictionary includes every variable in the cleaned P2 dataset. Missingness is calculated from the actual cleaned data.

In [24]:
dictionary = pd.DataFrame([
    {
        "Variable name": "film_year",
        "Type": "int",
        "Units": "year",
        "Factor levels": "N/A",
        "Valid range": "2015–2024 for this project",
        "Missingness": f'{clean_final["film_year"].isna().sum()} missing; missing would mean the film year could not be identified',
        "Notes": "Film year used by the Academy Award table and the project's within-year comparison."
    },
    {
        "Variable name": "award_year",
        "Type": "int",
        "Units": "year",
        "Factor levels": "N/A",
        "Valid range": "2016–2025 for this project",
        "Missingness": f'{clean_final["award_year"].isna().sum()} missing; derived from film_year',
        "Notes": "Ceremony calendar year, created as film_year + 1."
    },
    {
        "Variable name": "title",
        "Type": "string",
        "Units": "N/A",
        "Factor levels": "N/A",
        "Valid range": "A non-empty Best Picture nominee title",
        "Missingness": f'{clean_final["title"].isna().sum()} missing; missing would mean no usable film title was available',
        "Notes": "Human-readable film title; retained for reporting and later source matching."
    },
    {
        "Variable name": "title_clean",
        "Type": "string",
        "Units": "N/A",
        "Factor levels": "N/A",
        "Valid range": "Non-empty standardized title text",
        "Missingness": f'{clean_final["title_clean"].isna().sum()} missing; derived from title',
        "Notes": "Lowercase alphanumeric matching key. Original title is retained."
    },
    {
        "Variable name": "winner",
        "Type": "boolean",
        "Units": "N/A",
        "Factor levels": "True, False",
        "Valid range": "{True, False}",
        "Missingness": f'{clean_final["winner"].isna().sum()} missing',
        "Notes": "True identifies the Best Picture winner; validation requires exactly one winner per film year."
    }
])

dictionary

,Variable name,Type,Units,Factor levels,Valid range,Missingness,Notes
0,film_year,int,year,N/A,2015–2024 for this project,0 missing; missing would mean the film year co...,Film year used by the Academy Award table and ...
1,award_year,int,year,N/A,2016–2025 for this project,0 missing; derived from film_year,"Ceremony calendar year, created as film_year + 1."
2,title,string,N/A,N/A,A non-empty Best Picture nominee title,0 missing; missing would mean no usable film t...,Human-readable film title; retained for report...
3,title_clean,string,N/A,N/A,Non-empty standardized title text,0 missing; derived from title,Lowercase alphanumeric matching key. Original ...
4,winner,boolean,N/A,"True, False","{True, False}",0 missing,True identifies the Best Picture winner; valid...


### 4. Cleaning Log
The cleaning log records one row per decision. Each entry gives an exact affected count, explains why the decision was made, and states what information was lost. Because the original source is preserved separately, transformations to the analytical table can be reversed or reconstructed from the raw source and this log.

In [25]:
cleaning_log = pd.DataFrame(log_rows)
cleaning_log

,#,Column,Change made,Rows/cells affected,Why,What is lost
0,1,film_year,Filtered records to film years 2015–2024,554,The research question is explicitly limited to...,Award records outside the study period are exc...
1,2,title,Trimmed leading/trailing whitespace and collap...,0,Consistent title text reduces avoidable mismat...,Only non-substantive whitespace formatting; th...
2,3,film_year,Removed records without a usable four-digit fi...,0,A film year is required to define the study pe...,Any records lacking a usable film year would b...
3,4,title,Removed records without a usable film title,0,A title is required for the planned one-to-one...,Records without a usable film identifier would...
4,5,"film_year, title",Removed exact duplicate title-year records,81,Each nomination should appear once in the nomi...,Repeated copies of the same title-year record ...
5,6,award_year,Created ceremony year as film_year + 1,14,The ceremony generally occurs in the calendar ...,Nothing; this is a derived variable and film_y...
6,7,title_clean,Created lowercase alphanumeric standardized ti...,14,A separate standardized key will help resolve ...,Nothing from the source because the original t...
7,8,winner,Created Boolean winner indicator from the firs...,14,Wikipedia's Best Picture tables list the winni...,Nothing; winner status is derived and source t...
8,9,columns,Dropped source/helper fields from the final an...,3,The final P2 file keeps only variables needed ...,Source-table and raw parsing fields are not ca...
9,10,index,Reset pandas row index after filtering and cle...,14,A sequential index makes the cleaned table eas...,Only the temporary pandas index; no source var...


### 5. Row and Column Accounting
The accounting below reconciles the raw project extract with the cleaned analytical table. Filtering to the study period is shown separately because audit_raw includes records outside the project's population.

In [26]:
raw_project_rows = len(project_raw)
raw_project_cols = project_raw.shape[1]

rows_removed_exact_duplicates = exact_dupes
rows_removed_near_duplicates = 0
rows_removed_other = missing_year_rows + missing_title_rows

cleaned_rows = len(clean_final)

# project_raw has helper/raw fields; the final table also adds analytical variables.
raw_project_column_names = set(project_raw.columns)
clean_column_names = set(clean_final.columns)

columns_dropped = len(raw_project_column_names - clean_column_names)
columns_added = len(clean_column_names - raw_project_column_names)

accounting = pd.DataFrame({
    "Measure": [
        "Rows in raw project-period extract",
        "Rows removed as exact duplicates",
        "Rows removed as near-duplicates",
        "Rows removed for other reasons",
        "Rows in cleaned file",
        "Columns in raw project-period extract",
        "Columns dropped",
        "Columns added",
        "Columns in cleaned file"
    ],
    "Count": [
        raw_project_rows,
        rows_removed_exact_duplicates,
        rows_removed_near_duplicates,
        rows_removed_other,
        cleaned_rows,
        raw_project_cols,
        columns_dropped,
        columns_added,
        clean_final.shape[1]
    ]
})

display(accounting)

assert raw_project_rows - rows_removed_exact_duplicates - rows_removed_near_duplicates - rows_removed_other == cleaned_rows
assert raw_project_cols - columns_dropped + columns_added == clean_final.shape[1]

print("Accounting reconciles successfully.")

,Measure,Count
0,Rows in raw project-period extract,95
1,Rows removed as exact duplicates,81
2,Rows removed as near-duplicates,0
3,Rows removed for other reasons,0
4,Rows in cleaned file,14
5,Columns in raw project-period extract,4
6,Columns dropped,3
7,Columns added,4
8,Columns in cleaned file,5


Accounting reconciles successfully.


### 6. Provenance Brief
This dataset contains films nominated for the Academy Award for Best Picture, including the winning film for each year. The information comes from Wikipedia's Academy Award for Best Picture article, which compiles historical award results and identifies nominees and winners. For this project, I use films from the 2015–2024 film years because my research question compares IMDb audience ratings of Best Picture winners with the other nominees from the same years.
The dataset provides the information needed to identify which films competed for Best Picture and which film won each year. However, it does not contain the IMDb audience ratings or vote counts needed to answer the full research question. Those measures will come from IMDb and will be combined with this dataset in a later project milestone.
The data can support comparisons between winners and non-winning nominees, but it cannot explain why a film won or establish that winning Best Picture causes differences in audience ratings. Differences in film titles and release-year conventions may also create matching problems when the data is later combined with IMDb.

| Criterion | Weight | Self-Score | Evidence |
|---|---:|---:|---|
| **Diagnosis & Data Dictionary** | ×2 | **3/3** | Systematic audit and complete dictionary for all variables. |
| **Cleaning Execution** | ×2 | **3/3** | Cleaning decisions were appropriate and supported the analysis. |
| **Cleaning Log** | ×3 | **3/3** | Each decision includes counts, reasoning, and information lost. |
| **Provenance Brief** | ×1 | **3/3** | Clearly explains the dataset, source, purpose, and limitations. |
| **Tidy Structure & Reproducibility** | ×2 | **2/3** | Organized and reproducible; README and raw-data setup need confirmation. |
| **Weighted Total** | **×10** | **29/30** | |